# Multiclass Classification
Runs six-class, seven-class, and eight-class cervical cytology classification.


In [1]:
from pathlib import Path

# EDIT THIS: path to your local riva_1.0 folder
DATASET_PATH = r"C:\Users\Admin\Downloads\riva_1.0"

# Outputs are shared across notebooks so the Results notebook can load them later.
OUTPUT_ROOT = Path(DATASET_PATH).parent / "riva_outputs"
BASE_DIR = Path(DATASET_PATH)
IMG_DIR = BASE_DIR / "images"
ANNOT_FILE = BASE_DIR / "annotations" / "annotations.json"
MODEL_DIR_PATH = OUTPUT_ROOT / "saved_models"
RESULTS_DIR_PATH = OUTPUT_ROOT / "results"
FIG_DIR_PATH = OUTPUT_ROOT / "figures"
for p in [MODEL_DIR_PATH, RESULTS_DIR_PATH, FIG_DIR_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("Paths configured:")
print(f"  Dataset     : {BASE_DIR}  ({'OK' if BASE_DIR.exists() else 'MISSING - check DATASET_PATH'})")
print(f"  Images      : {IMG_DIR}  ({'OK' if IMG_DIR.exists() else 'MISSING'})")
print(f"  Annotations : {ANNOT_FILE}  ({'OK' if ANNOT_FILE.exists() else 'MISSING'})")
print(f"  Models out  : {MODEL_DIR_PATH}")
print(f"  Results out : {RESULTS_DIR_PATH}")


Paths configured:
  Dataset     : C:\Users\Admin\Downloads\riva_1.0  (OK)
  Images      : C:\Users\Admin\Downloads\riva_1.0\images  (OK)
  Annotations : C:\Users\Admin\Downloads\riva_1.0\annotations\annotations.json  (OK)
  Models out  : C:\Users\Admin\Downloads\riva_outputs\saved_models
  Results out : C:\Users\Admin\Downloads\riva_outputs\results


In [2]:
import os, json, random, time, datetime, gc
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SIZE = 224
BATCH_SIZE = 32
IMG_EXTS = ['.png', '.jpg', '.jpeg', '.bmp']

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti


In [3]:
BINARY_CLASSES = ['NON_LESION', 'PRECANCEROUS']
NUM_BINARY = 2
BINARY_MAPPING = {
    'SCC':'PRECANCEROUS', 'HSIL':'PRECANCEROUS', 'ASCH':'PRECANCEROUS',
    'LSIL':'PRECANCEROUS', 'ASCUS':'PRECANCEROUS',
    'INFL':'NON_LESION', 'ENDO':'NON_LESION', 'NILM':'NON_LESION'
}

CLASS_NAMES_6 = ['ASCUS', 'LSIL', 'ASCH', 'HSIL', 'SCC', 'INFL']
CLASS_NAMES_7 = ['ASCUS', 'LSIL', 'ASCH', 'HSIL', 'SCC', 'INFL', 'ENDO']
CLASS_NAMES_8 = ['ASCUS', 'LSIL', 'ASCH', 'HSIL', 'SCC', 'INFL', 'ENDO', 'NILM']
PRIORITY_6 = ['SCC', 'HSIL', 'ASCH', 'ASCUS', 'LSIL', 'INFL']
PRIORITY_7 = ['SCC', 'HSIL', 'ASCH', 'ASCUS', 'LSIL', 'INFL', 'ENDO']
PRIORITY_8 = ['SCC', 'HSIL', 'ASCH', 'ASCUS', 'LSIL', 'INFL', 'ENDO', 'NILM']

with open(ANNOT_FILE, 'r', encoding='utf-8') as f:
    riva_annots = json.load(f)

def normalise_label(label):
    lbl = str(label).strip().upper()
    if lbl in ('ENDOCERVICAL', 'ENDOCERVICAL CELL', 'ENDOCERVICAL CELLS'):
        return 'ENDO'
    if lbl in ('INFLAMMATION', 'INFLAMMATORY'):
        return 'INFL'
    return lbl

def _main_multiclass_label(raw_labels, class_names, priority):
    mc_labels = raw_labels & set(class_names)
    if not mc_labels:
        return None
    return next((p for p in priority if p in mc_labels), None)

dataset_items = []
for img_name, ann_data in riva_annots.items():
    raw_labels = set()
    for ann_kps in ann_data.values():
        if not isinstance(ann_kps, list):
            continue
        for kp in ann_kps:
            lbl = normalise_label(kp.get('keypointlabels', ''))
            if lbl in set(CLASS_NAMES_8):
                raw_labels.add(lbl)
    if not raw_labels:
        continue

    img_path = next((IMG_DIR / f"{img_name}{e}" for e in IMG_EXTS if (IMG_DIR / f"{img_name}{e}").exists()), None)
    if img_path is None:
        continue

    binary_set = {BINARY_MAPPING[l] for l in raw_labels if l in BINARY_MAPPING}
    label_vector = [1 if cls in binary_set else 0 for cls in BINARY_CLASSES]
    binary_label = 1 if 'PRECANCEROUS' in binary_set else 0

    mc6 = _main_multiclass_label(raw_labels, CLASS_NAMES_6, PRIORITY_6)
    mc7 = _main_multiclass_label(raw_labels, CLASS_NAMES_7, PRIORITY_7)
    mc8 = _main_multiclass_label(raw_labels, CLASS_NAMES_8, PRIORITY_8)

    dataset_items.append({
        'img_name': img_name,
        'image_path': str(img_path),
        'label_vector': label_vector,
        'binary_label': binary_label,
        'mc6_class_label': CLASS_NAMES_6.index(mc6) if mc6 is not None else None,
        'mc7_class_label': CLASS_NAMES_7.index(mc7) if mc7 is not None else None,
        'mc8_class_label': CLASS_NAMES_8.index(mc8) if mc8 is not None else None,
        'labels_present': sorted(raw_labels),
    })

print(f"Dataset items: {len(dataset_items)}")
print(f"Binary PRECANCEROUS: {sum(d['binary_label'] for d in dataset_items)}")
for version, names in [('6', CLASS_NAMES_6), ('7', CLASS_NAMES_7), ('8', CLASS_NAMES_8)]:
    key = f'mc{version}_class_label'
    counts = Counter(d[key] for d in dataset_items if d[key] is not None)
    print(f"\n{version}-class distribution:")
    for i, n in enumerate(names):
        print(f"  {n:<8}: {counts.get(i, 0):>5}")


Dataset items: 959
Binary PRECANCEROUS: 623

6-class distribution:
  ASCUS   :    99
  LSIL    :   173
  ASCH    :    51
  HSIL    :   190
  SCC     :   110
  INFL    :   202

7-class distribution:
  ASCUS   :    99
  LSIL    :   173
  ASCH    :    51
  HSIL    :   190
  SCC     :   110
  INFL    :   202
  ENDO    :    11

8-class distribution:
  ASCUS   :    99
  LSIL    :   173
  ASCH    :    51
  HSIL    :   190
  SCC     :   110
  INFL    :   202
  ENDO    :    11
  NILM    :   123


In [4]:
def _strat_key(labels_present):
    return 'PRECANCEROUS' if any(BINARY_MAPPING.get(l) == 'PRECANCEROUS' for l in labels_present) else 'NON_LESION'

strat_keys = [_strat_key(d['labels_present']) for d in dataset_items]
all_indices = list(range(len(dataset_items)))

def safe_split(X, y, test_size=0.3, rs=SEED):
    cnt = Counter(y)
    strat = y if cnt and min(cnt.values()) >= 2 else None
    return train_test_split(X, y, test_size=test_size, stratify=strat, random_state=rs)

train_idx, temp_idx, _, _ = safe_split(all_indices, strat_keys, test_size=0.3)
val_idx, test_idx, _, _ = safe_split(temp_idx, [strat_keys[i] for i in temp_idx], test_size=0.5)

mc6_train_idx = [i for i in train_idx if dataset_items[i]['mc6_class_label'] is not None]
mc6_val_idx   = [i for i in val_idx   if dataset_items[i]['mc6_class_label'] is not None]
mc6_test_idx  = [i for i in test_idx  if dataset_items[i]['mc6_class_label'] is not None]
mc7_train_idx = [i for i in train_idx if dataset_items[i]['mc7_class_label'] is not None]
mc7_val_idx   = [i for i in val_idx   if dataset_items[i]['mc7_class_label'] is not None]
mc7_test_idx  = [i for i in test_idx  if dataset_items[i]['mc7_class_label'] is not None]
mc8_train_idx = [i for i in train_idx if dataset_items[i]['mc8_class_label'] is not None]
mc8_val_idx   = [i for i in val_idx   if dataset_items[i]['mc8_class_label'] is not None]
mc8_test_idx  = [i for i in test_idx  if dataset_items[i]['mc8_class_label'] is not None]

print(f"Binary split  - Train:{len(train_idx)} Val:{len(val_idx)} Test:{len(test_idx)}")
print(f"6-class split - Train:{len(mc6_train_idx)} Val:{len(mc6_val_idx)} Test:{len(mc6_test_idx)}")
print(f"7-class split - Train:{len(mc7_train_idx)} Val:{len(mc7_val_idx)} Test:{len(mc7_test_idx)}")
print(f"8-class split - Train:{len(mc8_train_idx)} Val:{len(mc8_val_idx)} Test:{len(mc8_test_idx)}")


Binary split  - Train:671 Val:144 Test:144
6-class split - Train:581 Val:118 Test:126
7-class split - Train:589 Val:118 Test:129
8-class split - Train:671 Val:144 Test:144


In [5]:
imagenet_norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
train_tf = T.Compose([
    T.Resize((TARGET_SIZE, TARGET_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30),
    T.ColorJitter(0.3, 0.3, 0.3, 0.1),
    T.RandomAffine(10, translate=(0.1, 0.1)),
    T.ToTensor(), imagenet_norm
])
train_tf_lsil = T.Compose([
    T.Resize((TARGET_SIZE, TARGET_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30),
    T.ColorJitter(0.5, 0.5, 0.5, 0.2),
    T.RandomAutocontrast(p=0.4),
    T.RandomAffine(10, translate=(0.1, 0.1)),
    T.ToTensor(), imagenet_norm
])
val_tf = T.Compose([T.Resize((TARGET_SIZE, TARGET_SIZE)), T.ToTensor(), imagenet_norm])

def unnormalize(t, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]):
    img = t.clone()
    for c, m, s in zip(img, mean, std):
        c.mul_(s).add_(m)
    return img

class RIVADataset(Dataset):
    def __init__(self, items, indices, transform=None, task='binary', version='8'):
        self.items = items
        self.indices = indices
        self.transform = transform
        self.task = task
        self.version = str(version)
        self.label_key = {'6':'mc6_class_label', '7':'mc7_class_label', '8':'mc8_class_label'}.get(self.version, 'mc8_class_label')
        names = {'6':CLASS_NAMES_6, '7':CLASS_NAMES_7, '8':CLASS_NAMES_8}.get(self.version, CLASS_NAMES_8)
        self.lsil_idx = names.index('LSIL') if 'LSIL' in names else -1

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        item = self.items[self.indices[idx]]
        img = Image.open(item['image_path']).convert('RGB')
        if self.task != 'binary' and self.transform is train_tf and item.get(self.label_key) == self.lsil_idx:
            img = train_tf_lsil(img)
        elif self.transform:
            img = self.transform(img)
        if self.task == 'binary':
            return img, torch.tensor(item['label_vector'], dtype=torch.float32)
        return img, torch.tensor(item[self.label_key], dtype=torch.long)

def make_binary_loaders():
    return (
        DataLoader(RIVADataset(dataset_items, train_idx, train_tf, 'binary'), BATCH_SIZE, shuffle=True, num_workers=0),
        DataLoader(RIVADataset(dataset_items, val_idx, val_tf, 'binary'), BATCH_SIZE, shuffle=False, num_workers=0),
        DataLoader(RIVADataset(dataset_items, test_idx, val_tf, 'binary'), BATCH_SIZE, shuffle=False, num_workers=0),
        RIVADataset(dataset_items, train_idx, train_tf, 'binary'),
        RIVADataset(dataset_items, val_idx, val_tf, 'binary'),
        RIVADataset(dataset_items, test_idx, val_tf, 'binary'),
    )

def make_multiclass_loaders(version='8'):
    version = str(version)
    if version == '6':
        tr, v, te, key = mc6_train_idx, mc6_val_idx, mc6_test_idx, 'mc6_class_label'
    elif version == '7':
        tr, v, te, key = mc7_train_idx, mc7_val_idx, mc7_test_idx, 'mc7_class_label'
    else:
        tr, v, te, key = mc8_train_idx, mc8_val_idx, mc8_test_idx, 'mc8_class_label'
    train_ds = RIVADataset(dataset_items, tr, train_tf, 'multiclass', version)
    val_ds = RIVADataset(dataset_items, v, val_tf, 'multiclass', version)
    test_ds = RIVADataset(dataset_items, te, val_tf, 'multiclass', version)
    labels_tr = [dataset_items[i][key] for i in tr]
    class_counts = Counter(labels_tr)
    sample_weights = [1.0 / class_counts[l] for l in labels_tr]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    return (
        DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=0),
        DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=0),
        DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0),
        train_ds, val_ds, test_ds
    )


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

class ResNetModel(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    def forward(self, x):
        return self.backbone(x)

class EfficientNetModel(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    def forward(self, x):
        return self.backbone(x)

MODEL_REGISTRY = {'CNN': SimpleCNN, 'ResNet34': ResNetModel, 'EfficientNet': EfficientNetModel}

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            alpha = alpha / alpha.sum() * len(alpha)
        self.register_buffer('alpha', alpha)
    def forward(self, logits, targets):
        log_prob = nn.functional.log_softmax(logits, dim=1)
        prob = torch.exp(log_prob)
        log_pt = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = prob.gather(1, targets.unsqueeze(1)).squeeze(1).clamp(1e-6, 1.0 - 1e-6)
        loss = -((1.0 - pt) ** self.gamma) * log_pt
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean() if self.reduction == 'mean' else loss.sum()

def build_optimizer(model, lr=1e-4, wd=1e-4, opt_type='adam'):
    if opt_type.lower() == 'adamw':
        return optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train(); total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

def eval_loss(model, loader, criterion, device):
    model.eval(); total = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total += criterion(model(x), y).item()
    return total / max(len(loader), 1)

def format_duration(seconds):
    return f"{seconds:.1f}s" if seconds < 60 else f"{seconds / 60:.2f}m"

def current_lr(optimizer):
    return optimizer.param_groups[0]['lr']

def train_with_early_stopping(model, train_loader, val_loader, criterion,
                              lr=1e-4, wd=1e-4, opt_type='adam', epochs=30,
                              patience=7, device=device, run_label='Training'):
    model.to(device)
    optimizer = build_optimizer(model, lr, wd, opt_type)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    best_val, wait = float('inf'), 0
    history = {'train_loss': [], 'val_loss': [], 'epoch_seconds': [], 'learning_rate': []}
    for epoch in range(epochs):
        epoch_start = time.perf_counter()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = eval_loss(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_seconds = time.perf_counter() - epoch_start
        lr_now = current_lr(optimizer)
        history['train_loss'].append(float(train_loss))
        history['val_loss'].append(float(val_loss))
        history['epoch_seconds'].append(float(epoch_seconds))
        history['learning_rate'].append(float(lr_now))
        print(f"{run_label} | epoch {epoch + 1:03d}/{epochs:03d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={lr_now:.2e} | time={format_duration(epoch_seconds)}")
        if val_loss < best_val:
            best_val, wait = val_loss, 0
        else:
            wait += 1
            if wait >= patience:
                print(f"{run_label} | early_stopping_epoch={epoch + 1}")
                break
    return model, best_val, history

MC_SEARCH_SPACE = [
    {'lr': 1e-4, 'wd': 1e-4, 'dropout': 0.3, 'opt': 'adam'},
    {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.3, 'opt': 'adam'},
    {'lr': 1e-4, 'wd': 5e-4, 'dropout': 0.5, 'opt': 'adam'},
]


In [7]:
# Multiclass training setup
TUNE_EPOCHS = 20
FINAL_EPOCHS = 20
PATIENCE = 5
MULTICLASS_VERSIONS = {
    '6': CLASS_NAMES_6,
    '7': CLASS_NAMES_7,
    '8': CLASS_NAMES_8,
}

print("Multiclass classification setup")
print(f"versions: {list(MULTICLASS_VERSIONS.keys())}")
print(f"tuning_epochs: {TUNE_EPOCHS} | final_epochs: {FINAL_EPOCHS} | patience: {PATIENCE}")

def multiclass_indices(version):
    version = str(version)
    if version == '6':
        return mc6_train_idx, mc6_val_idx, mc6_test_idx, 'mc6_class_label'
    if version == '7':
        return mc7_train_idx, mc7_val_idx, mc7_test_idx, 'mc7_class_label'
    return mc8_train_idx, mc8_val_idx, mc8_test_idx, 'mc8_class_label'

def make_multiclass_criterion(version, indices):
    version = str(version)
    class_names = MULTICLASS_VERSIONS[version]
    key = f'mc{version}_class_label'
    labels = [dataset_items[i][key] for i in indices if dataset_items[i][key] is not None]
    counts = Counter(labels)
    weights = torch.tensor([1.0 / max(counts.get(i, 0), 1) for i in range(len(class_names))], dtype=torch.float32, device=device)
    return FocalLoss(gamma=2.0, alpha=weights)

def make_combined_multiclass_loader(version):
    train_indices, val_indices, _, key = multiclass_indices(version)
    combined_indices = train_indices + val_indices
    train_ds = RIVADataset(dataset_items, combined_indices, train_tf, 'multiclass', version)
    monitor_ds = RIVADataset(dataset_items, val_indices, val_tf, 'multiclass', version)
    labels = [dataset_items[i][key] for i in combined_indices]
    class_counts = Counter(labels)
    sample_weights = [1.0 / class_counts[l] for l in labels]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    return (
        DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=0),
        DataLoader(monitor_ds, BATCH_SIZE, shuffle=False, num_workers=0),
        train_ds,
        monitor_ds,
    )


Multiclass classification setup
versions: ['6', '7', '8']
tuning_epochs: 20 | final_epochs: 20 | patience: 5


In [8]:
# Multiclass tuning and evaluation

def tune_multiclass(version, model_class, model_name, train_loader, val_loader, criterion):
    class_names = MULTICLASS_VERSIONS[str(version)]
    print(f"\n{version}-class {model_name} tuning")
    best_model, best_cfg, best_val, best_history = None, None, float('inf'), None
    tuning_rows = []

    for cfg_num, cfg in enumerate(MC_SEARCH_SPACE, start=1):
        print(f"{version}-class {model_name} tuning config {cfg_num}/{len(MC_SEARCH_SPACE)} | cfg={cfg}")
        model = model_class(len(class_names), dropout=cfg['dropout'])
        model, val_loss, history = train_with_early_stopping(
            model, train_loader, val_loader, criterion,
            lr=cfg['lr'], wd=cfg['wd'], opt_type=cfg['opt'], epochs=TUNE_EPOCHS,
            patience=PATIENCE, run_label=f"{version}-class {model_name} tune cfg {cfg_num}/{len(MC_SEARCH_SPACE)}")
        tuning_rows.append({'version': version, 'model': model_name, 'config_id': cfg_num, 'cfg': cfg, 'best_val_loss': float(val_loss)})
        print(f"{version}-class {model_name} tuning config {cfg_num} complete | best_val_loss={val_loss:.4f}")

        if val_loss < best_val:
            best_model, best_cfg, best_val, best_history = model, cfg, val_loss, history
            print(f"{version}-class {model_name} new best | best_val_loss={best_val:.4f} | cfg={best_cfg}")

    torch.save({
        'model_state_dict': best_model.state_dict(),
        'model_architecture': model_name,
        'task': f'multiclass_{version}',
        'class_names': class_names,
        'best_cfg': best_cfg,
        'best_val_loss': best_val,
        'tuning_rows': tuning_rows,
        'history': best_history,
    }, MODEL_DIR_PATH / f"multiclass_{version}class_{model_name}_tuned_best.pth")
    return best_model, best_cfg, tuning_rows

def evaluate_multiclass(model, loader, class_names, verbose=True):
    model.eval(); model.to(device)
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(y.numpy().tolist())
    out = {
        'accuracy': float(np.mean(np.array(y_true) == np.array(y_pred))),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))).tolist(),
    }
    if verbose:
        print(f"Multiclass metrics | accuracy={out['accuracy']:.3f} | precision_macro={out['precision_macro']:.3f} | recall_macro={out['recall_macro']:.3f} | f1_macro={out['f1_macro']:.3f}")
    return out

def final_multiclass(version, model_class, model_name, cfg, criterion, test_loader):
    class_names = MULTICLASS_VERSIONS[str(version)]
    print(f"\n{version}-class {model_name} final training")
    print(f"{version}-class {model_name} final config | cfg={cfg}")
    combined_loader, monitor_loader, _, _ = make_combined_multiclass_loader(version)
    model = model_class(len(class_names), dropout=cfg['dropout'])
    model, final_val, history = train_with_early_stopping(
        model, combined_loader, monitor_loader, criterion,
        lr=cfg['lr'], wd=cfg['wd'], opt_type=cfg['opt'], epochs=FINAL_EPOCHS,
        patience=PATIENCE, run_label=f"{version}-class {model_name} final")
    torch.save(model.state_dict(), MODEL_DIR_PATH / f"multiclass_{version}class_{model_name}_final.pth")
    print(f"{version}-class {model_name} final complete | best_val_loss={final_val:.4f}")
    metrics = evaluate_multiclass(model, test_loader, class_names)
    return model, metrics, history


In [9]:
# Run multiclass experiments
all_multiclass_outputs = {}

for version, class_names in MULTICLASS_VERSIONS.items():
    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = make_multiclass_loaders(version)
    criterion = make_multiclass_criterion(version, multiclass_indices(version)[0])
    print(f"\n{version}-class split | train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)} | classes={class_names}")

    tuned_cfgs = {}
    tuning_rows = []
    trained_models = {}
    results = {}
    histories = {}

    for cls, name in [(SimpleCNN, 'CNN'), (ResNetModel, 'ResNet34'), (EfficientNetModel, 'EfficientNet')]:
        _, cfg, rows = tune_multiclass(version, cls, name, train_loader, val_loader, criterion)
        tuned_cfgs[name] = cfg
        tuning_rows.extend(rows)

    for cls, name in [(SimpleCNN, 'CNN'), (ResNetModel, 'ResNet34'), (EfficientNetModel, 'EfficientNet')]:
        model, metrics, history = final_multiclass(version, cls, name, tuned_cfgs[name], criterion, test_loader)
        trained_models[name] = model
        results[name] = metrics
        histories[name] = history

    best_name = max(results, key=lambda n: (results[n]['f1_macro'], results[n]['accuracy']))
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    payload = {
        'task': f'multiclass_{version}',
        'timestamp': timestamp,
        'classes': class_names,
        'best_model': best_name,
        'selection_metric': 'macro F1 then accuracy',
        'search_space': MC_SEARCH_SPACE,
        'tuning_epochs': TUNE_EPOCHS,
        'final_epochs': FINAL_EPOCHS,
        'patience': PATIENCE,
        'loss': 'FocalLoss gamma=2.0 with inverse-frequency class weighting',
        'tuning_rows': tuning_rows,
        'results': results,
    }
    out_path = RESULTS_DIR_PATH / f"multiclass_{version}class_results_{timestamp}.json"
    with open(out_path, 'w') as f:
        json.dump(payload, f, indent=2)

    all_multiclass_outputs[version] = payload
    print(f"{version}-class best model: {best_name}")
    print(f"{version}-class saved results: {out_path}")



6-class split | train=581 | val=118 | test=126 | classes=['ASCUS', 'LSIL', 'ASCH', 'HSIL', 'SCC', 'INFL']

6-class CNN tuning
6-class CNN tuning config 1/3 | cfg={'lr': 0.0001, 'wd': 0.0001, 'dropout': 0.3, 'opt': 'adam'}
6-class CNN tune cfg 1/3 | epoch 001/020 | train_loss=1.1991 | val_loss=1.0142 | lr=1.00e-04 | time=25.7s
6-class CNN tune cfg 1/3 | epoch 002/020 | train_loss=1.1211 | val_loss=1.0369 | lr=1.00e-04 | time=25.9s
6-class CNN tune cfg 1/3 | epoch 003/020 | train_loss=1.1150 | val_loss=1.0522 | lr=1.00e-04 | time=25.8s
6-class CNN tune cfg 1/3 | epoch 004/020 | train_loss=1.1044 | val_loss=1.0476 | lr=1.00e-04 | time=25.7s
6-class CNN tune cfg 1/3 | epoch 005/020 | train_loss=1.1135 | val_loss=1.0591 | lr=5.00e-05 | time=25.8s
6-class CNN tune cfg 1/3 | epoch 006/020 | train_loss=1.1094 | val_loss=1.0601 | lr=5.00e-05 | time=30.6s
6-class CNN tune cfg 1/3 | early_stopping_epoch=6
6-class CNN tuning config 1 complete | best_val_loss=1.0142
6-class CNN new best | best_val